In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from mlxtend.frequent_patterns import apriori, association_rules
import shap
import joblib

path='./Datasets/Raw/'
df = pd.read_csv(path+'survey.csv')

print("The first five rows of data:")
print(df.head())

print("\nALL field names:")
print(df.columns.tolist())

The first five rows of data:
         Timestamp  Age  Gender         Country state self_employed  \
0  2014/8/27 11:29   37  Female   United States    IL           NaN   
1  2014/8/27 11:29   44       M   United States    IN           NaN   
2  2014/8/27 11:29   32    Male          Canada   NaN           NaN   
3  2014/8/27 11:29   31    Male  United Kingdom   NaN           NaN   
4  2014/8/27 11:30   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often           6月25日  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely           6月25日  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

                leave mental_health_consequence phys_health_consequence  \
0       Somewhat easy                        No                      No 

In [2]:
# df = pd.read_csv("C:/Users/lenovo/Data mining--project/cleaned_survey_data.csv")
df = pd.read_csv("./Datasets/Processed/cleaned_survey_data.csv")
# 1.Employee Turnover Risk Labels
# work_interfere---employee turnover risk labels：
# "often"--1 , "sometimes"&"never"&"rarely"--0
df["churn_risk"] = df["work_interfere"].apply(lambda x: 1 if x in ["Often", "Sometimes"] else 0)

# 2.mental health problems labels----treatment
# treatment----"yes"--1 ,"no"--0
df["mental_issue"] = df["treatment"].apply(lambda x: 1 if x == "Yes" else 0)

In [3]:
# feature coding（文字转数字）
# 要编码的分类型特征（除了Age和两个目标变量）
encode_cols = ["Gender", "family_history", "remote_work", "benefits", "care_options"]

In [4]:
# 独热编码（避免模型误解“类别顺序”，比如认为“Male”>“Female”）
encoder = OneHotEncoder(drop="first")  # drop="first"避免冗余
encoded_features = encoder.fit_transform(df[encode_cols])

In [5]:
# 编码后的特征转为DataFrame，方便合并
encoded_df = pd.DataFrame(
    encoded_features.toarray(),
    columns=encoder.get_feature_names_out(encode_cols),  # 列名示例：Gender_Female, family_history_Yes
    index=df.index
)
print("编码后DataFrame形状：", encoded_df.shape)

编码后DataFrame形状： (1250, 8)


In [6]:
# 合并数值特征（Age）和编码后的分类型特征
num_features = df[["Age"]]
final_features = pd.concat([num_features, encoded_df], axis=1)
print("合并后特征形状：", final_features.shape)

# 合并两个目标变量
final_df = pd.concat([final_features, df[["churn_risk", "mental_issue"]]], axis=1)
print("最终数据形状：", final_df.shape)

# 查看最终用于分析的数据
print("\n最终分析数据形状：", final_df.shape)
print("最终数据前5行：")
print(final_df.head())

# 保存特征工程后的数据
final_df.to_csv("final_analysis_data.csv", index=False)
print("\n特征工程后的数据已保存为 final_analysis_data.csv")

合并后特征形状： (1250, 9)
最终数据形状： (1250, 11)

最终分析数据形状： (1250, 11)
最终数据前5行：
   Age  Gender_Male  Gender_Other  family_history_Yes  remote_work_Yes  \
0   37          0.0           0.0                 0.0              0.0   
1   44          1.0           0.0                 0.0              0.0   
2   32          1.0           0.0                 0.0              0.0   
3   31          1.0           0.0                 1.0              0.0   
4   31          1.0           0.0                 0.0              1.0   

   benefits_No  benefits_Yes  care_options_Not sure  care_options_Yes  \
0          0.0           1.0                    1.0               0.0   
1          0.0           0.0                    0.0               0.0   
2          1.0           0.0                    0.0               0.0   
3          1.0           0.0                    0.0               1.0   
4          0.0           1.0                    0.0               0.0   

   churn_risk  mental_issue  
0           1    